In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path
from typing import Iterable, Optional

# --- CONFIG ---
JSON_ROOT = Path(r"schemas/json")
SIMAL_ROOT = Path(r"schemas/simal")

# File extensions to include per mode
JSON_EXTENSIONS = {".json", ".txt"}
SIMAL_EXTENSIONS = {".simal", ".txt"}

# Skip suffixes / names
SKIP_SUFFIXES = {".input_log", ".log"}
SKIP_NAMES = {"execution.log", "untitled"}

# Tokenizers
GPT_TIKTOKEN_MODEL = "gpt-5"  # fallback handled if unavailable in tiktoken build
GEMMA_REPO_ID = "google/gemma-3-27b-it"

# Output
OUTPUT_DIR = Path(r"outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("JSON_ROOT:", JSON_ROOT)
print("SIMAL_ROOT:", SIMAL_ROOT)
print("OUTPUT_DIR:", OUTPUT_DIR)

In [ ]:
# --- Dependency check ---
import importlib
import subprocess

REQUIRED = ["tiktoken", "tokenizers", "huggingface_hub", "pandas"]
missing = [m for m in REQUIRED if importlib.util.find_spec(m) is None]

if missing:
    print("Missing packages:", missing)
    print("Install in this notebook kernel with:")
    print("  pip install " + " ".join(missing))
else:
    print("All required packages are available.")

In [ ]:
import pandas as pd
import tiktoken
from huggingface_hub import hf_hub_download
from tokenizers import Tokenizer

def load_gpt_encoding(model_name: str):
    """Returns a tiktoken Encoding; uses a robust fallback."""
    try:
        return tiktoken.encoding_for_model(model_name)
    except Exception:
        return tiktoken.get_encoding("o200k_base")

def load_gemma_tokenizer(repo_id: str) -> Tokenizer:
    """Load Gemma tokenizer via HF Hub tokenizer.json using `tokenizers` (no transformers)."""
    local_path = hf_hub_download(repo_id=repo_id, filename="tokenizer.json", token="<YOUR_HF_TOKEN>")
    return Tokenizer.from_file(local_path)

gpt_enc = load_gpt_encoding(GPT_TIKTOKEN_MODEL)
gemma_tok = load_gemma_tokenizer(GEMMA_REPO_ID)

print("Loaded GPT encoding:", getattr(gpt_enc, "name", "<unknown>"))
print("Loaded Gemma tokenizer from:", GEMMA_REPO_ID)

In [ ]:
def should_skip_path(dirpath: str) -> bool:
    parts = Path(dirpath).parts
    if any(p.startswith(".") for p in parts):
        return True
    if "__pycache__" in parts:
        return True
    # skip any path that contains a literal repo folder
    if "repo" in [p.lower() for p in parts]:
        return True
    return False

def iter_schema_files(root: Path, mode: str) -> Iterable[Path]:
    exts = JSON_EXTENSIONS if mode == "json" else SIMAL_EXTENSIONS
    for dirpath, _, filenames in os.walk(root):
        if should_skip_path(dirpath):
            continue
        for filename in filenames:
            if filename in SKIP_NAMES:
                continue
            p = Path(dirpath) / filename
            if p.suffix in SKIP_SUFFIXES:
                continue
            if p.suffix.lower() not in exts:
                continue
            yield p

def count_tokens(text: str, gpt_encoding, gemma_tokenizer: Tokenizer) -> tuple[int, int]:
    gpt_n = len(gpt_encoding.encode(text))
    gemma_n = len(gemma_tokenizer.encode(text).ids)
    return gpt_n, gemma_n

def scan_root(root: Path, mode: str):
    rows = []
    for fp in iter_schema_files(root, mode):
        rel = fp.relative_to(root)
        project = rel.parts[0] if rel.parts else "<unknown>"
        try:
            text = fp.read_text(encoding="utf-8")
        except UnicodeDecodeError:
            text = fp.read_text(encoding="utf-8", errors="replace")
        gpt_n, gemma_n = count_tokens(text, gpt_enc, gemma_tok)
        rows.append({
            "mode": mode,
            "project": project,
            "file": str(rel).replace("\\", "/"),
            "bytes": fp.stat().st_size,
            "tokens_gpt": gpt_n,
            "tokens_gemma": gemma_n,
        })
    return pd.DataFrame(rows)

df_json = scan_root(JSON_ROOT, "json")
df_simal = scan_root(SIMAL_ROOT, "simal")
df_all = pd.concat([df_json, df_simal], ignore_index=True)

print("Rows:", len(df_all))
df_all.head(10)

In [ ]:
# --- Aggregations ---
def agg_project(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame(columns=["mode","project","files","bytes","tokens_gpt","tokens_gemma"])
    return (
        df.groupby(["mode", "project"], as_index=False)
          .agg(files=("file", "count"), bytes=("bytes", "sum"), tokens_gpt=("tokens_gpt", "sum"), tokens_gemma=("tokens_gemma", "sum"), avg_tokens_gpt=("tokens_gpt", "mean"), avg_tokens_gemma=("tokens_gemma", "mean"))
    )

proj = agg_project(df_all)
proj.head(10)

In [ ]:
# --- Project-level JSON vs SiMAL comparison (match by project folder name) ---
json_proj = proj[proj["mode"] == "json"].drop(columns=["mode"]).rename(columns={
    "files": "files_json",
    "bytes": "bytes_json",
    "tokens_gpt": "tokens_gpt_json",
    "tokens_gemma": "tokens_gemma_json",
    "avg_tokens_gpt": "avg_tokens_gpt_json",
    "avg_tokens_gemma": "avg_tokens_gemma_json",
})
simal_proj = proj[proj["mode"] == "simal"].drop(columns=["mode"]).rename(columns={
    "files": "files_simal",
    "bytes": "bytes_simal",
    "tokens_gpt": "tokens_gpt_simal",
    "tokens_gemma": "tokens_gemma_simal",
    "avg_tokens_gpt": "avg_tokens_gpt_simal",
    "avg_tokens_gemma": "avg_tokens_gemma_simal",
})

cmp = json_proj.merge(simal_proj, on="project", how="outer")
for col in ["files_json","bytes_json","tokens_gpt_json","tokens_gemma_json", "avg_tokens_gpt_json", "avg_tokens_gemma_json", "files_simal","bytes_simal","tokens_gpt_simal","tokens_gemma_simal", "avg_tokens_gpt_simal", "avg_tokens_gemma_simal"]:
    cmp[col] = cmp[col].fillna(0).astype(int)

def add_deltas(df: pd.DataFrame, left: str, right: str, prefix: str):
    df[f"{prefix}_abs"] = df[right] - df[left]
    df[f"{prefix}_ratio"] = df.apply(lambda r: (r[right] / r[left]) if r[left] else (float('inf') if r[right] else 1.0), axis=1)
    df[f"{prefix}_pct"] = df.apply(lambda r: ((r[right] - r[left]) / r[left]) if r[left] else (float('inf') if r[right] else 0.0), axis=1)
    return df

cmp = add_deltas(cmp, "tokens_gpt_json", "tokens_gpt_simal", "gpt")
cmp = add_deltas(cmp, "tokens_gemma_json", "tokens_gemma_simal", "gemma")
cmp = add_deltas(cmp, "avg_tokens_gpt_json", "avg_tokens_gpt_simal", "avg_gpt")
cmp = add_deltas(cmp, "avg_tokens_gemma_json", "avg_tokens_gemma_simal", "avg_gemma")

cmp_sorted = cmp.sort_values("gpt_pct", ascending=False)
cmp_sorted.head(20)

In [ ]:
# --- Global totals (all dirs aggregated by mode) ---
global_totals = (
    df_all.groupby("mode", as_index=False)
         .agg(files=("file","count"), bytes=("bytes","sum"), tokens_gpt=("tokens_gpt","sum"), tokens_gemma=("tokens_gemma","sum"), avg_tokens_gpt=("tokens_gpt", "mean"), avg_tokens_gemma=("tokens_gemma", "mean"))
         .sort_values("mode")
)
global_totals

In [ ]:
# --- Save CSV outputs ---
df_all_out = OUTPUT_DIR / "simal_vs_json_schema_tokens_per_file.csv"
proj_out = OUTPUT_DIR / "simal_vs_json_schema_tokens_per_project.csv"
cmp_out = OUTPUT_DIR / "simal_vs_json_schema_tokens_json_vs_simal.csv"
global_out = OUTPUT_DIR / "simal_vs_json_schema_tokens_global_totals.csv"

df_all.to_csv(df_all_out, index=False)
proj.to_csv(proj_out, index=False)
cmp.to_csv(cmp_out, index=False)
global_totals.to_csv(global_out, index=False)

print("Wrote:")
print(" -", df_all_out)
print(" -", proj_out)
print(" -", cmp_out)
print(" -", global_out)